In [ ]:
import h5py
import numpy as np
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras import layers, models

# Load dataset
with h5py.File("https://cernbox.cern.ch/s/zUvpkKhXIp0MJ0g", "r") as f:
    X = f["X_jet"][:]   # shape: (N, 125, 125, 4)
    y = f["am"][:]      # target mass

# Select only ECAL (channel 3) and Track pT (channel 0)
X = X[:,:,:, [0,3]]

# Train-test split (80/20)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
model = models.Sequential([
    layers.Conv2D(32, (3,3), activation='relu', input_shape=(125,125,2)),
    layers.MaxPooling2D((2,2)),
    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D((2,2)),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dense(1)  # regression output
])

model.compile(optimizer='adam', loss='mse', metrics=['mae'])
history = model.fit(X_train, y_train, validation_split=0.1, epochs=20, batch_size=32)

In [ ]:
loss, mae = model.evaluate(X_test, y_test)
print("Test MAE:", mae)

In [ ]:
import tf2onnx

onnx_model, _ = tf2onnx.convert.from_keras(model, opset=13)
with open("sample.onnx", "wb") as f:
    f.write(onnx_model.SerializeToString())